# Pengujian TensorFlow Serving — Wine Quality

Notebook ini menguji deployment TensorFlow Serving di Railway melalui endpoint metadata, predict, dan Prometheus metrics.

In [ ]:
import base64
import requests
import tensorflow as tf

BASE_URL = 'https://wine-quality-mlops-production.up.railway.app'
MODEL_NAME = 'wine-quality'
FEATURE_KEYS = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
    'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide',
    'density', 'pH', 'sulphates', 'alcohol',
]

def serialize_example(features):
    tf_features = {
        key: tf.train.Feature(float_list=tf.train.FloatList(value=[float(features[key])]))
        for key in FEATURE_KEYS
    }
    example = tf.train.Example(features=tf.train.Features(feature=tf_features))
    return base64.b64encode(example.SerializeToString()).decode('utf-8')

def predict(features):
    payload = {'instances': [{'examples': {'b64': serialize_example(features)}}]}
    response = requests.post(
        f'{BASE_URL}/v1/models/{MODEL_NAME}:predict', json=payload, timeout=60
    )
    response.raise_for_status()
    return response.json()

## Metadata model (bukti deployment TF Serving)

In [ ]:
response = requests.get(f'{BASE_URL}/v1/models/{MODEL_NAME}', timeout=30)
print('Status:', response.status_code)
print(response.json())

## Prediksi TF Serving

In [ ]:
wine_sample = {
    'fixed_acidity': 7.4, 'volatile_acidity': 0.28, 'citric_acid': 0.34,
    'residual_sugar': 1.2, 'chlorides': 0.045, 'free_sulfur_dioxide': 35.0,
    'total_sulfur_dioxide': 141.0, 'density': 0.9940, 'pH': 3.42,
    'sulphates': 0.68, 'alcohol': 12.5,
}
result = predict(wine_sample)
print(result)

## Metrik Prometheus TF Serving

In [ ]:
response = requests.get(f'{BASE_URL}/monitoring/prometheus/metrics', timeout=30)
print('Status:', response.status_code)
for line in response.text.splitlines():
    if 'tensorflow' in line and not line.startswith('#'):
        print(line)